# Chapter 5 · Density Functional Theory — live notebook

This notebook runs entirely in your browser via the Pyodide kernel.
All state is saved to your browser's local storage; nothing is sent to a server.

Back to the chapter: <https://dongzhaohe321418-lab.github.io/materials-simulation-handbook/ch05-dft/>

A toy self-consistent-field loop in 1D. We pretend the exchange-correlation functional is a simple LDA-like local term and watch the density converge under linear mixing.

Only Pyodide-compatible packages are used (numpy, scipy, matplotlib, ipywidgets).


## A miniature 1D Kohn–Sham SCF

Two electrons in a soft Coulombic external well. The Hartree term is computed from the running density by direct integration; XC is approximated by $v_{xc}(r) = -c \rho^{1/3}$ — a 1D caricature of LDA. Watch the convergence of the total density.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 300
L = 14.0
x = np.linspace(-L / 2, L / 2, N)
dx = x[1] - x[0]

def kinetic_matrix(n, dx):
    pref = 0.5 / dx ** 2
    return np.diag(2 * pref * np.ones(n)) - np.diag(pref * np.ones(n - 1), 1) \
        - np.diag(pref * np.ones(n - 1), -1)

def soft_coulomb(x, alpha=1.0, x0=0.0):
    return -1.0 / np.sqrt((x - x0) ** 2 + alpha ** 2)

v_ext = soft_coulomb(x)
T = kinetic_matrix(N, dx)

def trapz(y, x):
    return float(0.5 * np.sum((y[1:] + y[:-1]) * np.diff(x)))

def hartree(rho, x):
    # 1D soft Coulomb interaction
    return np.array([trapz(rho / np.sqrt((x - xi) ** 2 + 1.0), x) for xi in x])

def vxc(rho, c=0.5):
    return -c * np.cbrt(np.clip(rho, 1e-12, None))

# Initial guess: ground state of the bare external potential
H0 = T + np.diag(v_ext)
vals, vecs = np.linalg.eigh(H0)
psi = vecs[:, 0] / np.sqrt(dx)
rho = 2.0 * psi ** 2  # two electrons, spin-paired

history = [rho.copy()]
mix = 0.3
for it in range(40):
    v_eff = v_ext + hartree(rho, x) + vxc(rho)
    H = T + np.diag(v_eff)
    vals, vecs = np.linalg.eigh(H)
    psi_new = vecs[:, 0] / np.sqrt(dx)
    rho_new = 2.0 * psi_new ** 2
    delta = np.max(np.abs(rho_new - rho))
    rho = mix * rho_new + (1 - mix) * rho
    history.append(rho.copy())
    if delta < 1e-6:
        break
print(f'converged in {it + 1} iterations, eigenvalue = {vals[0]:.4f} Ha')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for k, r in enumerate(history[:: max(1, len(history) // 6)]):
    ax.plot(x, r, label=f'iter {k * max(1, len(history) // 6)}', alpha=0.7)
ax.set_xlabel('x (bohr)')
ax.set_ylabel('density rho(x)')
ax.set_title('SCF convergence under linear mixing')
ax.legend(fontsize=8)
plt.show()


Experiment: shrink `mix` to 0.1 — convergence slows. Push it past 1.0 and you see oscillations or divergence, exactly the pathology DIIS and Pulay mixing are designed to avoid.